# Pycytominer Processing — v3 Case Overview

Normalizes, aggregates, and applies feature selection to SingleSlice outputs from `1_FeatureSorting.ipynb` (`FeaturesImages_150526_none`). Results saved to `1_Data/results/sections/`.

---

## Acquisitions

Two plate barcodes (`PB100082`, `PB100083`), each imaged across 6 sectants in 7 conditions. Acquisitions are identified by `Metadata_image_id` (the analysis_run id).

|              | Cols 1–8           | Cols 9–16            | Cols 17–24           |
|--------------|--------------------|----------------------|----------------------|
| **Rows A–H** | Sectant 1 — nyquist | Sectant 2 — double_dens | Sectant 3 — 37C     |
| **Rows I–P** | Sectant 4 — noclear ⚠️ | Sectant 5 — 20_40 (subset) | Sectant 6 — 10x **and** 40x |

Sectant 6 has two acquisitions (10x and 40x) on the **same wells**, distinguishable only by `Metadata_image_id`.

| image_id | plate    | condition   | z-slices |
|----------|----------|-------------|----------|
| 8787     | PB100082 | nyquist     | 44       |
| 8804     | PB100083 | nyquist     | 44       |
| 8780     | PB100082 | double_dens | 14       |
| 8799     | PB100083 | double_dens | 14       |
| 8783     | PB100082 | 37C         | 14       |
| 8802     | PB100083 | 37C         | 14       |
| 8793     | PB100082 | noclear     | 14       |
| 8795     | PB100083 | noclear     | 14       |
| 8789     | PB100082 | 20_40       | 14       |
| 8791     | PB100082 | 20_40       | 14       |
| 8797     | PB100083 | 20_40       | 14       |
| 8827     | PB100082 | 10x         | 14       |
| 8831     | PB100083 | 10x         | 14       |
| 8825     | PB100082 | 40x         | 14       |
| 8829     | PB100083 | 40x         | 14       |

---

## Output cases (pooled across plates per condition)

| Case               | Conditions   | Z-slices                                       | Output file                          |
|--------------------|--------------|------------------------------------------------|--------------------------------------|
| nyquist_all        | nyquist      | all 44                                         | `selected_nyquist_all.parquet`       |
| nyquist_14planes   | nyquist      | 14 evenly spaced (0..43)                       | `selected_nyquist_14planes.parquet`  |
| double_dens        | double_dens  | all 14                                         | `selected_double_dens.parquet`       |
| 37C                | 37C          | all 14                                         | `selected_37C.parquet`               |
| noclear ⚠️         | noclear      | all 14                                         | `selected_noclear.parquet`           |
| 20_40              | 20_40        | all 14                                         | `selected_20_40.parquet`             |
| 10x                | 10x          | all 14                                         | `selected_10x.parquet`               |
| 40x                | 40x          | all 14                                         | `selected_40x.parquet`               |

**Notes:**
- Normalization is per `(Metadata_Barcode, Metadata_z)` — plate batch effects are removed even when pooling both plates.
- Section 4 / `noclear` is uncleared — segmentation may be unreliable.
- `nyquist_14planes` allows fair comparison with the other 14-slice conditions; `nyquist_all` retains the full oversampled resolution.


In [ ]:
# --- repo path bootstrap (added by the port) ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, external, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

# Pycytominer
from pycytominer import feature_select
from pycytominer import normalize
# from pycytominer import aggregate

# Set current working directory


In [ ]:
def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))

    return list_of_selected_features, list_of_metadata

In [ ]:
cell_line  = 'HCT116'
data_dir   = str(features("exp3_clearing_mag_z", "150526", "SingleSlice")) + "/"
output_dir = profiles("exp3_clearing_mag_z", "sections/")
os.makedirs(output_dir, exist_ok=True)

# image_id (analysis_run id) -> condition
IMG_TO_CONDITION = {
    # nyquist (44 z-slices, sectant 1)
    8787: 'nyquist',      # PB100082
    8804: 'nyquist',      # PB100083
    # double_dens (sectant 2)
    8780: 'double_dens',  # PB100082
    8799: 'double_dens',  # PB100083
    # 37C (sectant 3)
    8783: '37C',          # PB100082
    8802: '37C',          # PB100083
    # noclear (sectant 4) — uncleared
    8793: 'noclear',      # PB100082
    8795: 'noclear',      # PB100083
    # 20_40 (sectant 5) — well subset; PB100082 split across two image_ids
    8789: '20_40',        # PB100082
    8791: '20_40',        # PB100082
    8797: '20_40',        # PB100083
    # sectant 6 — same wells, two magnifications
    8827: '10x',          # PB100082
    8831: '10x',          # PB100083
    8825: '40x',          # PB100082
    8829: '40x',          # PB100083
}

In [ ]:
files = [f for f in os.listdir(data_dir) if cell_line in f and 'MedianAgg' in f]
data = pd.concat([pd.read_parquet(data_dir + f) for f in files], ignore_index=True)
data['_col'] = data['Metadata_Well'].str[1:].astype(int)
data['Metadata_condition'] = data['Metadata_image_id'].map(IMG_TO_CONDITION)

unmapped = data[data['Metadata_condition'].isna()]['Metadata_image_id'].unique()
if len(unmapped):
    print(f'WARNING: {len(unmapped)} image_id(s) not in IMG_TO_CONDITION: {sorted(unmapped)}')

print(f'Loaded {data.shape[0]} rows x {data.shape[1]} columns')
print('Plates:', data['Metadata_Barcode'].unique().tolist())
print()
print('Condition x plate cross-tab (n rows):')
print(pd.crosstab(data['Metadata_condition'], data['Metadata_Barcode']))
print()
print('Z-slice count per condition:')
print(data.groupby('Metadata_condition')['Metadata_z'].nunique())

In [ ]:
def process_case(data, case_name, conditions=None, z_slices=None):
    df = data.copy()
    if conditions is not None:
        df = df[df['Metadata_condition'].isin(conditions)]
    if z_slices is not None:
        df = df[df['Metadata_z'].isin(z_slices)]
    if df.empty:
        print(f'[{case_name}] No data after filtering — skipping.')
        return None
    print(f'[{case_name}] {df["Metadata_Barcode"].nunique()} plate(s), '
          f'{df["Metadata_z"].nunique()} z-slice(s), {df["Metadata_Well"].nunique()} well(s)')

    # Normalize per plate x z-slice (so plate batch effects are removed even when pooling plates)
    df['Metadata_plate_slice'] = df['Metadata_Barcode'] + '_' + df['Metadata_z'].astype(str)
    features_list = list_features(df)[0]
    normalized_parts = []
    for unit in df['Metadata_plate_slice'].unique():
        temp = df[df['Metadata_plate_slice'] == unit]
        if temp[temp['Metadata_cmpdname'] == 'dmso'].empty:
            print(f'  [{case_name}] No DMSO in {unit} — skipping slice.')
            continue
        norm_temp = normalize(temp, features=features_list, image_features=False,
                              meta_features='infer',
                              samples="Metadata_cmpdname == 'dmso'",
                              method='standardize')
        normalized_parts.append(norm_temp)
    if not normalized_parts:
        print(f'[{case_name}] No slices could be normalized — skipping.')
        return None
    normalized = pd.concat(normalized_parts, ignore_index=True)

    # Aggregate across z-slices (median per well, within plate)
    features_list = list_features(normalized)[0]
    drop_from_meta = ['Metadata_z', 'Metadata_PlateWell', 'Metadata_plate_slice', '_col']
    meta_cols = [c for c in normalized.columns if c not in features_list and c not in drop_from_meta]
    aggregated = normalized.groupby(['Metadata_Barcode', 'Metadata_PlateWell']).agg(
        {**{c: 'first'  for c in meta_cols if c not in ('Metadata_Barcode',)},
         **{c: 'median' for c in features_list}}
    ).reset_index()

    # Feature selection + clipping
    to_clip = feature_select(aggregated, features=list_features(aggregated)[0],
                             operation=['variance_threshold', 'correlation_threshold', 'drop_na_columns'])
    selected = pd.concat([
        to_clip[list_features(to_clip)[1]],
        to_clip[list_features(to_clip)[0]].clip(lower=-40, upper=40, axis=1)
    ], axis=1)

    out_path = f'{output_dir}selected_{case_name}.parquet'
    selected.to_parquet(out_path)
    print(f'  -> saved {selected.shape[0]} wells x {selected.shape[1]} cols to {out_path}')
    return selected

## Nyquist (sectant 1) — high z-resolution

In [ ]:
# All 44 z-slices
process_case(data, 'nyquist_all', conditions=['nyquist'])

In [ ]:
# 14 evenly-spaced z-slices (0..43) — matched z-count to other conditions for fair comparison
z_14 = np.linspace(0, 43, 14).round().astype(int).tolist()
print('nyquist 14-plane selection:', z_14)
process_case(data, 'nyquist_14planes', conditions=['nyquist'], z_slices=z_14)

## Other conditions (14 z-slices each, both plates pooled)

In [ ]:
process_case(data, 'double_dens', conditions=['double_dens'])

In [ ]:
process_case(data, '37C', conditions=['37C'])

In [ ]:
process_case(data, 'noclear', conditions=['noclear'])

In [ ]:
# Well subset only (see plate layout); PB100082 split across two image_ids — both mapped to '20_40'
process_case(data, '20_40', conditions=['20_40'])

In [ ]:
process_case(data, '10x', conditions=['10x'])

In [ ]:
process_case(data, '40x', conditions=['40x'])